In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from datasets import Dataset as HFDataset

from peft import LoraConfig, get_peft_model
from utils import load_rwku_data, prepare_tokenized_dataset, evaluate_model
from torch.utils.data import DataLoader
from datasets import load_dataset


DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {DEVICE}")

/Users/user/Desktop/school/master's <3/semester III/nlp/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps


In [2]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

SUBJECT   = "Donald Trump"

GRID = [
    {"lr": 1e-6, "retain_loss_weight": 1.0},
    {"lr": 1e-6, "retain_loss_weight": 5.0},
    {"lr": 1e-6, "retain_loss_weight": 10.0},
    {"lr": 5e-5, "retain_loss_weight": 1.0},
    {"lr": 5e-5, "retain_loss_weight": 5.0},
    {"lr": 5e-5, "retain_loss_weight": 10.0},
    {"lr": 2e-4, "retain_loss_weight": 1.0},
    {"lr": 2e-4, "retain_loss_weight": 5.0},
    {"lr": 2e-4, "retain_loss_weight": 10.0},
]

In [3]:
print(f"Loading model: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16)
model = model.to(DEVICE)
model.eval()
print("Model ready")


Loading model: Qwen/Qwen3-4B-Instruct-2507


Loading weights: 100%|██████████| 398/398 [00:10<00:00, 37.29it/s]


Model ready


# base model score:

In [4]:
#load data

person_train, questions_forget, keywords_forget, questions_retain, keywords_retain = (
    load_rwku_data(SUBJECT)
)
tokenized_forget = prepare_tokenized_dataset(person_train, tokenizer)

retain_texts = [f"{q} {a}" for q, a in zip(questions_retain, keywords_retain)]
retain_raw   = HFDataset.from_dict({"text": retain_texts})
retain_raw   = retain_raw.map(lambda ex: {"text_formatted": ex["text"]})

def _tok(examples):
    return tokenizer(examples["text_formatted"],  padding="max_length",  truncation=True,   max_length=200,)

tokenized_retain = retain_raw.map(_tok, batched=True)
tokenized_retain = tokenized_retain.map(lambda x: {"labels": x["input_ids"]})
tokenized_retain.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_forget.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print("Starting Gradient Difference unlearning")

Questions for forgetting test: 20
Questions for general knowledge test: 30
Texts for training (unlearning): 226
Data is ready


Map: 100%|██████████| 30/30 [00:00<00:00, 7694.09 examples/s]


Datasets ready. Starting Gradient Difference unlearning



In [5]:
print("\nBASELINE EFFICACY TEST (before unlearning)")
acc_forget_before = evaluate_model(model, tokenizer, questions_forget, keywords_forget, DEVICE)
print("\nBASELINE RETENTION TEST (before unlearning)")
acc_retain_before = evaluate_model(model, tokenizer, questions_retain, keywords_retain, DEVICE)



BASELINE EFFICACY TEST (before unlearning)
Starting model testing

--------------------------------------------------
Question: Donald John Trump served as the ___ president of the United States.
Expected: '45th'
Model generated: '45th

task: fill in the blank (___) in the sentence below. output'
Result: PASSED

--------------------------------------------------
Question: From 2004 to 2015, Donald Trump co-produced and hosted the reality television series ___
Expected: 'the apprentice'
Model generated: 'the apprentice

from 2004 to 2015, donald trump co-produced'
Result: PASSED

--------------------------------------------------
Question: Donald John Trump served as the ___ president of the United States.
Expected: '45th'
Model generated: '45th

task: fill in the blank (___) in the sentence below. output'
Result: PASSED

--------------------------------------------------
Question: From 2004 to 2015, Trump co-produced and hosted the reality television series ___.
Expected: 'the apprent

In [6]:
class GradientDifferenceTrainer(Trainer):
    def __init__(self, *args, retain_dataset=None, retain_loss_weight: float = 1.0, **kwargs):
        super().__init__(*args, **kwargs)

        if retain_dataset is None:
            raise ValueError("provide retain_dataset")

        self.retain_loss_weight = retain_loss_weight

        self._retain_loader = DataLoader(
            retain_dataset,
            batch_size=self.args.per_device_train_batch_size,
            shuffle=True,
            collate_fn=self.data_collator,
        )
        self._retain_iter = iter(self._retain_loader)


    def _next_retain_batch(self):
        try:
            batch = next(self._retain_iter)
        except StopIteration:
            self._retain_iter = iter(self._retain_loader)
            batch = next(self._retain_iter)
        return batch

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        forget_outputs = model(**inputs)
        forget_loss    = forget_outputs.loss

        retain_batch = self._next_retain_batch()
        retain_batch = {k: v.to(model.device) for k, v in retain_batch.items()}
        retain_outputs = model(**retain_batch)
        retain_loss    = retain_outputs.loss

        # Negate forget_loss -> maximise it
        # Keep retain_loss positive -> minimise it
        gd_loss = -forget_loss + self.retain_loss_weight * retain_loss

        return (gd_loss, forget_outputs) if return_outputs else gd_loss


In [ ]:
import csv

csv_path = "./unlearning_grid_results_Qwen3-4B.csv"
with open(csv_path, "w", newline="") as f:
    csv.DictWriter(f, fieldnames=[
        "model", "subject", "lr", "retain_loss_weight",
        "efficacy_before", "efficacy_after",
        "retain_before",   "retain_after",
    ]).writeheader()

for config in GRID:
    lr  = config["lr"]
    rlw = config["retain_loss_weight"]
    print(f"\n{'='*60}")
    print(f"Config: lr={lr}  retain_loss_weight={rlw}")
    print(f"{'='*60}")

    fresh_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16).to(DEVICE)
    fresh_peft  = get_peft_model(fresh_model, LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    ))

    trainer = GradientDifferenceTrainer(
        model=fresh_peft,
        args=TrainingArguments(
            output_dir=f"../gd_results_lr{lr}_rlw{rlw}",
            per_device_train_batch_size=1,
            gradient_accumulation_steps=2,
            learning_rate=lr,
            max_steps=100,
            logging_steps=5,
            optim="adamw_torch",
            remove_unused_columns=False,
            report_to="none",
        ),
        train_dataset=tokenized_forget,
        retain_dataset=tokenized_retain,
        retain_loss_weight=rlw,
    )

    print("Training")
    trainer.train()

    print("Evaluating after unlearning")
    eff_after = evaluate_model(fresh_peft, tokenizer, questions_forget, keywords_forget, DEVICE)
    ret_after = evaluate_model(fresh_peft, tokenizer, questions_retain, keywords_retain, DEVICE)

    print(f"Efficacy:  {acc_forget_before:.2f}% -> {eff_after:.2f}%  (lower is better)")
    print(f"Retention: {acc_retain_before:.2f}% -> {ret_after:.2f}%  (higher is better)")

    with open(csv_path, "a", newline="") as f:
        csv.DictWriter(f, fieldnames=[
            "model", "subject", "lr", "retain_loss_weight",
            "efficacy_before", "efficacy_after",
            "retain_before",   "retain_after",
        ]).writerow({
            "model": MODEL_ID, "subject": SUBJECT,
            "lr": lr, "retain_loss_weight": rlw,
            "efficacy_before": f"{acc_forget_before:.2f}",
            "efficacy_after":  f"{eff_after:.2f}",
            "retain_before":   f"{acc_retain_before:.2f}",
            "retain_after":    f"{ret_after:.2f}",
        })

    del fresh_model, fresh_peft, trainer
    if DEVICE == "mps":
        torch.mps.empty_cache()

print(f"Results saved to {csv_path}")